# **Projet Steam: bloc2 Big Data**

## **Part2: EDA AND VIZUALISATION**

In [0]:
from pyspark.sql import functions as F

# 1. Rechargement du fichier JSON brut depuis le S3 d'Ubisoft
path = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"
df_raw = spark.read.json(path)

# 2. Re-création dynamique du schéma aplati (le flatten validé en Partie 1)
tag_columns = [f"data.tags.`{c}`" for c in df_raw.select("data.tags.*").columns]

df_clean_base = df_raw.select(
    "data.appid",
    "data.name",
    "data.developer",
    "data.publisher",
    "data.price",
    "data.initialprice",
    "data.discount",
    "data.positive",
    "data.negative",
    "data.genre",
    "data.languages",
    "data.categories",
    "data.release_date",
    "data.required_age",
    "data.short_description",
    "data.owners",
    "data.ccu",
    "data.header_image",
    "data.type",
    "data.website",
    
    # Extraction des plateformes
    "data.platforms.windows",
    "data.platforms.mac",
    "data.platforms.linux",
    
    # Déballage des colonnes de tags
    *tag_columns
)

print("🚀 Environnement initialisé avec succès ! ")

🚀 Environnement initialisé avec succès ! 


## **Analysis at the "macro" level**

Q1: Which publisher has released the most games on Steam?

In [0]:
from pyspark.sql import functions as F

# 1. On s'assure d'éliminer les éditeurs nuls ou vides pour ne pas fausser le Top
df_publishers = (
    df_clean_base
    .filter(F.col("publisher").isNotNull() & (F.col("publisher") != ""))
    .groupBy("publisher")
    .count()
    .orderBy(F.col("count").desc())
)

# 2. Remplacement de .show(5) par un .limit(10) combiné au display() natif
display(df_publishers.limit(10))

publisher,count
Big Fish Games,422
8floor,202
SEGA,165
Strategy First,151
Square Enix,141
Choice of Games,140
Sekai Project,132
HH-Games,132
Ubisoft,127
Laush Studio,126


Databricks visualization. Run in Databricks to view.

_Q1 answer: Big Fish Games is the one who published the most on STEAM._

Q2: What are the best rated games?

In [0]:
df_most_positive = (
    df_clean_base
    .orderBy(F.col("positive").desc())
)

df_most_positive.select(
    "name", "positive", "negative"
).show(5, truncate=False)

display(df_most_positive.select("name", "positive", "negative").limit(5))

+--------------------------------+--------+--------+
|name                            |positive|negative|
+--------------------------------+--------+--------+
|Counter-Strike: Global Offensive|5943345 |787093  |
|Dota 2                          |1534895 |317916  |
|Grand Theft Auto V              |1229265 |213379  |
|PUBG: BATTLEGROUNDS             |1185361 |908515  |
|Terraria                        |1014711 |22380   |
+--------------------------------+--------+--------+
only showing top 5 rows


name,positive,negative
Counter-Strike: Global Offensive,5943345,787093
Dota 2,1534895,317916
Grand Theft Auto V,1229265,213379
PUBG: BATTLEGROUNDS,1185361,908515
Terraria,1014711,22380


_Q2 answer: the best rated game is "Counter-Strike: Global Offensive"._

Q3: Are there years with more releases? Were there more or fewer game releases during the Covid, for example?

In [0]:
from pyspark.sql import functions as F

from pyspark.sql import functions as F

# 1. Ta logique de parsing
df_dates = df_clean_base.withColumn(
    "release_date_parsed",
    F.coalesce(
        F.try_to_date("release_date", "yyyy/MM/dd"),
        F.try_to_date("release_date", "yyyy/M/d"),
        F.try_to_date("release_date", "yyyy/MM"),
        F.try_to_date("release_date", "yyyy/M"),
        F.try_to_date("release_date", "yyyy")
    )
)

# 2. Extraction de l'année + Nettoyage des dates aberrantes (Crucial pour Ubisoft)
df_years = df_dates.withColumn(
    "release_year",
    F.year("release_date_parsed")
).filter(
    (F.col("release_year").isNotNull()) & 
    (F.col("release_year") >= 2015) & # On cible la période moderne
    (F.col("release_year") <= 2026)   # On évite les bugs de dates futures
)

# 3. Agrégation ET Tri par ANNÉE
df_releases_by_year = (
    df_years.groupBy("release_year")
            .count()
            .orderBy("release_year") # Tri chronologique indispensable
)

# 4. Affichage dans le notebook
display(df_releases_by_year)

release_year,count
2015,2576
2016,4185
2017,6017
2018,7678
2019,6968
2020,8305
2021,8823
2022,7455


Databricks visualization. Run in Databricks to view.

_Q3 answer: we have the most realeses during the covid between 2020 and 2022._

Q4: How are the prizes distributed? Are there many games with a discount?

In [0]:
from pyspark.sql import functions as F

# ==========================================
# 1. NETTOYAGE ET SÉCURISATION DES DONNÉES
# ==========================================
# On convertit le prix en Float/Double et la réduction en Entier.
# Les valeurs malformées ou nulles sont transformées en 0.
df_prices_clean = df_clean_base.withColumn(
    "clean_price", F.coalesce(F.col("price").cast("double"), F.lit(0.0))
).withColumn(
    "clean_discount", F.coalesce(F.col("discount").cast("int"), F.lit(0))
)

# ==========================================
# 2. DISTRIBUTION DES PRIX (Price Distribution)
# ==========================================
# Pour éviter un graphique illisible, on catégorise les prix par tranches business
df_price_brackets = df_prices_clean.withColumn(
    "price_bracket",
    F.when(F.col("clean_price") == 0.0, "01. Free-to-Play")
     .when((F.col("clean_price") > 0.0) & (F.col("clean_price") <= 5.0), "02. Under $5")
     .when((F.col("clean_price") > 5.0) & (F.col("clean_price") <= 10.0), "03. $5 to $10")
     .when((F.col("clean_price") > 10.0) & (F.col("clean_price") <= 20.0), "04. $10 to $20")
     .when((F.col("clean_price") > 20.0) & (F.col("clean_price") <= 50.0), "05. $20 to $50")
     .otherwise("06. AAA Games ($50+)")
)

# Agrégation pour la distribution des prix
distribution_prix = (
    df_price_brackets.groupBy("price_bracket")
    .count()
    .orderBy("price_bracket")
)

print("--- Distribution des tranches de prix ---")
display(distribution_prix)

# ==========================================
# 3. ANALYSE DES RÉDUCTIONS (Discounts)
# ==========================================
# Un jeu est considéré en promotion si son discount est strictement supérieur à 0
df_discount_status = df_prices_clean.withColumn(
    "has_discount",
    F.when(F.col("clean_discount") > 0, "With Discount")
     .otherwise("No Discount")
)

total_games = df_clean_base.count()

analyse_discounts = (
    df_discount_status.groupBy("has_discount")
    .agg(
        F.count("name").alias("game_count"),
        F.round((F.count("name") / total_games) * 100, 2).alias("percentage")
    )
)

print("--- Proportion de jeux en promotion ---")
display(analyse_discounts)

--- Distribution des tranches de prix ---


price_bracket,count
01. Free-to-Play,7780
05. $20 to $50,413
06. AAA Games ($50+),47498


Databricks visualization. Run in Databricks to view.

--- Proportion de jeux en promotion ---


has_discount,game_count,percentage
No Discount,53173,95.48
With Discount,2518,4.52


_Q4 answer: the majority of games cost more than 50$. We have also 4.5% of games with discount._

Q5: What are the most represented languages?

In [0]:
from pyspark.sql import functions as F

# 1. On découpe la chaîne de caractères au niveau des virgules pour recréer un ARRAY
df_language_array = df_clean_base.withColumn("language_list", F.split(F.col("languages"), ", "))

# 2. Maintenant qu'on a un ARRAY, on peut utiliser explode() en toute sécurité
df_languages_exploded = df_language_array.withColumn("single_language", F.explode(F.col("language_list")))

# 3. Agrégation et tri
df_language_counts = (
    df_languages_exploded.groupBy("single_language")
    .count()
    .orderBy(F.col("count").desc())
)

display(df_language_counts.limit(10))

single_language,count
English,55116
German,14019
French,13426
Russian,12922
Simplified Chinese,12782
Spanish - Spain,12233
Japanese,10368
Italian,9304
Portuguese - Brazil,6750
Korean,6599


Databricks visualization. Run in Databricks to view.

_Q5 answer: the most represented langauge is "English"._

Q6: Are there many games prohibited for children under 16/18?

In [0]:
from pyspark.sql import functions as F

# 1. Extraction numérique et Cast sécurisé avec l'API standard des colonnes Spark
# Si 'required_age' contient 'MA 15+', regexp_extract isole '15'. 
# S'il contient du texte pur (ex: 'Unrated'), le cast en 'int' renverra automatiquement NULL.
df_safe_age = df_clean_base.withColumn(
    "clean_required_age",
    F.regexp_extract(F.col("required_age"), r"(\d+)", 1).cast("int")
)

# 2. Création de la colonne conditionnelle
df_age_categories = df_safe_age.withColumn(
    "age_rating",
    F.when(F.col("clean_required_age") >= 18, "Prohibited under 18")
     .when(F.col("clean_required_age") >= 16, "Prohibited under 16")
     .when(F.col("clean_required_age") > 0, "Other restricted age")
     .otherwise("All audiences / Unknown")
)

# 3. Calcul des volumes et pourcentages globaux pour Ubisoft
total_games = df_clean_base.count()

df_age_analysis = (
    df_age_categories.groupBy("age_rating")
    .agg(
        F.count("name").alias("game_count"),
        F.round((F.count("name") / total_games) * 100, 2).alias("percentage")
    )
    .orderBy(F.col("game_count").desc())
)

# 4. Affichage final
display(df_age_analysis)

age_rating,game_count,percentage
All audiences / Unknown,55030,98.81
Other restricted age,355,0.64
Prohibited under 18,230,0.41
Prohibited under 16,76,0.14


_Q6 answer: not too much games are prohibited under 16/18, we have 0.41% prohibited under 18 years and 0.14% are prohibited under 16 years._

## **Genres analysis**

Q7: What are the most represented genres?

In [0]:
df_genre_popularity = (
    df_clean_base
    .groupBy("genre")
    .count()
    .orderBy(F.col("count").desc())
)

display(df_genre_popularity.limit(15))

genre,count
"Action, Indie",3460
"Casual, Indie",3060
"Action, Adventure, Indie",2783
"Adventure, Indie",2316
"Action, Casual, Indie",1914
"Adventure, Casual, Indie",1811
Indie,1756
Action,1633
Casual,1433
Adventure,1021


Databricks visualization. Run in Databricks to view.

_Q7 answer: "Action, Indie" is the most represented genre._

Q8: Are there any genres that have a better positive/negative review ratio?

In [0]:
from pyspark.sql import functions as F

# 1. Nettoyage : On sépare les genres combinés et on les "explose" (une ligne par genre unitaire)
df_genres_exploded = df_clean_base.withColumn(
    "genre_exploded",
    F.explode(F.split(F.col("genre"), ",\\s*"))
)

# 2. Calcul du score individuel et application du seuil de confiance
df_ratings_prepared = (
    df_genres_exploded
    .withColumn("rating_score", 
                (F.col("positive") / (F.col("positive") + F.col("negative"))) * 100)
    .filter((F.col("positive") + F.col("negative")) > 50)
)

# 3. Agrégation sur le genre unitaire explosé
df_genre_ratings_clean = (
    df_ratings_prepared
    .groupBy("genre_exploded")
    .agg(F.round(F.avg("rating_score"), 2).alias("avg_positive_percentage"))
    .orderBy(F.col("avg_positive_percentage").desc())
)

# 4. Affichage interactif pour Databricks
display(df_genre_ratings_clean.limit(15))

genre_exploded,avg_positive_percentage
Game Development,83.69
Photo Editing,80.93
Web Publishing,80.06
Design & Illustration,79.46
Animation & Modeling,78.69
Adventure,78.13
Casual,78.09
Indie,77.77
Sexual Content,76.84
RPG,76.62


Databricks visualization. Run in Databricks to view.

_Q8 answer: "Game Development" is the better positive/negative review ratio_

Q9: Do some publishers have favorite genres?

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Séparation et explosion des genres composites pour travailler sur des genres unitaires
df_genres_exploded = df_clean_base.withColumn(
    "single_genre",
    F.explode(F.split(F.col("genre"), ",\\s*"))
)

# 2. Compter le nombre de jeux pour chaque combinaison Éditeur / Genre unitaire
# On applique le filtre pour éliminer le bruit des éditeurs manquants ou vides
df_publisher_genre_counts = (
    df_genres_exploded
    .filter(F.col("publisher").isNotNull() & (F.col("publisher") != ""))
    .groupBy("publisher", "single_genre")
    .agg(F.count("name").alias("game_count"))
)

# 3. Fenêtre partitionnée par éditeur et triée par volume décroissant
window_spec = Window.partitionBy("publisher").orderBy(F.col("game_count").desc())

# 4. Attribution du rang pour isoler le genre N°1 (le favori)
df_ranked_genres = (
    df_publisher_genre_counts
    .withColumn("rank", F.rank().over(window_spec))
    .filter(F.col("rank") == 1)
)

# 5. Filtrage sur les éditeurs significatifs (ex: minimum 15 jeux dans leur genre favori)
# afin d'obtenir un visuel macro lisible et pertinent pour le jury
df_top_publishers_favorites = (
    df_ranked_genres
    .filter(F.col("game_count") >= 15)
    .orderBy(F.col("game_count").desc())
)

# 6. Affichage optimisé pour Databricks
display(df_top_publishers_favorites.limit(20))

publisher,single_genre,game_count,rank
Big Fish Games,Casual,418,1
8floor,Casual,202,1
Choice of Games,RPG,139,1
HH-Games,Casual,132,1
Laush Studio,Indie,124,1
Alawar Entertainment,Casual,105,1
Sekai Project,Casual,99,1
Sokpop Collective,Indie,97,1
Slitherine Ltd.,Strategy,96,1
Reforged Group,Indie,88,1


_Q9 answer: for example the publisher "Big Fish Games" has "Casual" as favorite genre._

Q10: What are the most lucrative genres?

In [0]:
from pyspark.sql import functions as F

# 1. Calcul du volume proxy de ventes au niveau du jeu individuel
df_with_sales_proxy = df_clean_base.withColumn(
    "total_reviews", 
    F.col("positive") + F.col("negative")
)

# 2. Estimation du Chiffre d'Affaires brut par jeu (coalesce pour gérer les jeux gratuits/nulls)
df_with_revenue = df_with_sales_proxy.withColumn(
    "estimated_revenue", 
    F.coalesce(F.col("price").cast("double"), F.lit(0.0)) * F.col("total_reviews")
)

# 3. CRUCIAL : On sépare et explose les genres composites pour travailler sur des entités unitaires
df_genres_exploded = df_with_revenue.withColumn(
    "single_genre",
    F.explode(F.split(F.col("genre"), ",\\s*"))
)

# 4. Agrégation sur le genre unitaire propre
df_lucrative_genres_clean = (
    df_genres_exploded.groupBy("single_genre")
    .agg(
        F.round(F.sum("estimated_revenue"), 2).alias("total_estimated_revenue"),
        F.round(F.avg("price"), 2).alias("average_genre_price"),
        F.sum("total_reviews").alias("total_volume_proxy")
    )
    .orderBy(F.col("total_estimated_revenue").desc())
)

# 5. Affichage pour ton tableau de bord Databricks
display(df_lucrative_genres_clean.limit(5))

single_genre,total_estimated_revenue,average_genre_price,total_volume_proxy
Action,1.05306097551E11,772.71,64546277
Adventure,6.782266358E10,800.62,35342598
Indie,5.0648939283E10,656.81,36772257
RPG,4.92395177E10,904.27,22699856
Simulation,3.4066895559E10,909.16,17972902


Databricks visualization. Run in Databricks to view.

_Q10 answer: "Action" is the most lucrative genre._

## **Platform analysis**

Q11: Are most games available on Windows/Mac/Linux instead?

In [0]:
from pyspark.sql import functions as F

# 1. Sélection et conversion des colonnes de plateformes présentes à la racine
# On utilise F.coalesce pour transformer les valeurs manquantes/nulles en 0
df_platforms_clean = df_clean_base.select(
    F.col("name"),
    F.coalesce(F.col("windows").cast("int"), F.lit(0)).alias("is_windows"),
    F.coalesce(F.col("mac").cast("int"), F.lit(0)).alias("is_mac"),
    F.coalesce(F.col("linux").cast("int"), F.lit(0)).alias("is_linux")
)

# 2. Récupération du nombre total de jeux pour calculer les pourcentages
total_games = df_clean_base.count()

# 3. Agrégation macro pour calculer les volumes et les pourcentages de disponibilité
df_platform_distribution = df_platforms_clean.agg(
    F.sum("is_windows").alias("Total_Windows"),
    F.sum("is_mac").alias("Total_Mac"),
    F.sum("is_linux").alias("Total_Linux")
).withColumn(
    "Pct_Windows", F.round((F.col("Total_Windows") / total_games) * 100, 2)
).withColumn(
    "Pct_Mac", F.round((F.col("Total_Mac") / total_games) * 100, 2)
).withColumn(
    "Pct_Linux", F.round((F.col("Total_Linux") / total_games) * 100, 2)
)

# 4. Affichage du résultat pour votre graphique Databricks
display(df_platform_distribution)

Total_Windows,Total_Mac,Total_Linux,Pct_Windows,Pct_Mac,Pct_Linux
55676,12770,8458,99.97,22.93,15.19


Databricks visualization. Run in Databricks to view.

_Q11 answer: majority of games are available in Windows but not too much in Mac and Linux._

Q12: Do certain genres tend to be preferentially available on certain platforms?

In [0]:
from pyspark.sql import functions as F

# 1. Séparation et explosion des genres composites pour travailler sur des genres unitaires
df_genres_exploded = df_clean_base.withColumn(
    "single_genre",
    F.explode(F.split(F.col("genre"), ",\\s*"))
)

# 2. Calcul des pourcentages de compatibilité par plateforme pour chaque genre unitaire
df_genre_platform_trends_clean = (
    df_genres_exploded
    .groupBy("single_genre")
    .agg(
        F.count("name").alias("total_games_in_genre"),
        F.round(F.avg(F.col("windows").cast("int")) * 100, 2).alias("pct_windows"),
        F.round(F.avg(F.col("mac").cast("int")) * 100, 2).alias("pct_mac"),
        F.round(F.avg(F.col("linux").cast("int")) * 100, 2).alias("pct_linux")
    )
    # Seuil de confiance : on élimine les genres marginaux de moins de 10 jeux
    .filter(F.col("total_games_in_genre") > 10)
    .orderBy(F.col("total_games_in_genre").desc())
)

# 3. Affichage des résultats pour le tableau de bord Databricks
display(df_genre_platform_trends_clean.limit(20))

single_genre,total_games_in_genre,pct_windows,pct_mac,pct_linux
Indie,39681,99.99,25.04,17.59
Action,23759,99.98,19.21,14.22
Casual,22086,99.98,23.23,14.96
Adventure,21431,99.98,23.51,15.41
Strategy,10895,99.97,27.58,16.76
Simulation,10836,99.96,22.51,14.14
RPG,9534,99.99,23.58,15.98
Early Access,6145,100.0,14.65,10.28
Free to Play,3393,99.94,24.9,13.97
Sports,2666,99.96,18.98,10.77


Databricks visualization. Run in Databricks to view.